# PitVision — Notebook 2: Model Training & Evaluation

In [ ]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

from xgboost import XGBRegressor

In [ ]:

df = pd.read_parquet("../data/features/features.parquet")

FEATURES = ["TyreLife", "tyre_life_squared", "race_progress", "Stint",
            "compound_SOFT", "compound_MEDIUM", "compound_HARD"]
X = df[FEATURES]
y = df["LapTimeDelta"]

print("Feature matrix shape:", X.shape)

Feature matrix shape: (3750, 7)


## Train/test split

We hold back 20% of the data the model never sees during training. We grade the
model on that unseen 20%, that's the only honest way to estimate real-world
accuracy. `random_state=42` makes the split reproducible (same result every run).

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("Training laps:", len(X_train))
print("Testing laps:", len(X_test))

Training laps: 3000
Testing laps: 750


## Train all three models and compare

We loop over the three models, train each, and print MAE (average error in
seconds — lower is better) and R2 (how much of the variation it explains —
higher is better, max 1.0).

In [4]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    results[name] = (model, mae, r2)
    print(f"{name:18s}  MAE={mae:.3f}s   R2={r2:.3f}")

Linear Regression   MAE=0.678s   R2=0.537
Random Forest       MAE=0.647s   R2=0.547
XGBoost             MAE=0.612s   R2=0.600


### Check against your target tiers (Section 10.2)

| Tier | MAE | R2 |
|------|-----|-----|
| Acceptable | <= 0.80 s | >= 0.65 |
| Good | <= 0.50 s | >= 0.80 |
| Excellent | <= 0.35 s | >= 0.90 |

## Sanity check 1: feature importance

This shows which inputs the model relies on most. Tire age and race progress
*should* dominate. If a compound column dominates instead, investigate.

In [5]:
rf = results["Random Forest"][0]
importance = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(importance)

Stint                0.500435
race_progress        0.250811
TyreLife             0.101381
tyre_life_squared    0.100209
compound_HARD        0.020099
compound_MEDIUM      0.018225
compound_SOFT        0.008842
dtype: float64


## Sanity check 2: cross-validation

5-fold cross-validation re-runs the test on 5 different splits. If the average
error here is close to the test MAE above, the model is stable (not just lucky).

In [6]:
cv = cross_val_score(
    RandomForestRegressor(n_estimators=200, random_state=42),
    X, y, cv=5, scoring="neg_mean_absolute_error")
print(f"Cross-validated MAE: {-cv.mean():.3f}s  (+/- {cv.std():.3f})")

Cross-validated MAE: 0.916s  (+/- 0.114)


## Save the best model

We pick the model with the lowest MAE and save it together with the exact feature order, so the dashboard loads the same setup it was trained on.

In [ ]:
best_name = min(results, key=lambda k: results[k][1])  
best_model = results[best_name][0]

joblib.dump({"model": best_model, "features": FEATURES},
            "../models/tire_degradation.pkl")
print(f"Saved best model: {best_name} -> ../models/tire_degradation.pkl")

Saved best model: XGBoost -> ../models/tire_degradation.pkl


**Week 2 milestone reached:** trained model saved to disk.

Quick test that inference works (run this last):

In [ ]:
import sys
sys.path.append("..")   
from src.inference import predict_lap_time

print("Example prediction (MEDIUM, 10 laps old, mid-race, stint 1):")
print(round(predict_lap_time("MEDIUM", 10, 0.5, 1), 2), "seconds")

Example prediction (MEDIUM, 10 laps old, mid-race, stint 1):
88.9 seconds
